<a href="https://colab.research.google.com/github/Khalidsyfullah/USplitVQA/blob/main/Direct_Model_Apply/New_Custom_Centralized_PATH_VQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets", "openpyxl", "tqdm"])
import re
import os, random, time
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR = "/kaggle/working/"; os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# CONFIG
# =============================================================================
D = 256
VOCAB_SIZE = 30522
MAX_SEQ = 64
HEADS = 4
DROP = 0.25
CBAM_BLOCKS = 3
TEXT_ENC_LAYERS = 2
TEXT_REFINE_LAYERS = 2
FUSE_LAYERS = 4
EPOCHS = 30; BS = 32; LR = 3e-4; WD = 1e-4; LS = 0.0
ES_PAT = 16; LR_PAT = 4; LR_FAC = 0.5; MIN_LR = 1e-6; VAL_SPLIT = 0.15


# ─────────────────────────────────────────────────────────────────────
# 2. PathVQA  (flaviagiammarino/path-vqa)
# ─────────────────────────────────────────────────────────────────────
# ~32,799 QA pairs · 4,998 pathology images
# Yes/no (~50%), plus open-ended: what/where/when/whose/how/how many
# Very diverse open-ended answers (organs, stains, cell types, diseases)

def normalize_answer_pathvqa(ans: str) -> str:
    """Normalize PathVQA answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true',
               'yes it is', 'yes, it is', 'affirmative'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not present', 'no it is not'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Plural→singular for common pathology terms ──
    plural_map = {
        'cells': 'cell', 'nuclei': 'nucleus', 'tissues': 'tissue',
        'arteries': 'artery', 'veins': 'vein', 'vessels': 'vessel',
        'glands': 'gland', 'nodes': 'node', 'lymph nodes': 'lymph node',
        'tumors': 'tumor', 'tumours': 'tumor', 'tumour': 'tumor',
        'lesions': 'lesion', 'cysts': 'cyst',
        'follicles': 'follicle', 'tubules': 'tubule',
        'neurons': 'neuron', 'fibers': 'fiber', 'fibres': 'fiber',
        'fibre': 'fiber', 'muscles': 'muscle',
        'kidneys': 'kidney', 'lungs': 'lung', 'bones': 'bone',
        'lymphocytes': 'lymphocyte', 'macrophages': 'macrophage',
        'neutrophils': 'neutrophil', 'erythrocytes': 'erythrocyte',
        'platelets': 'platelet',
    }
    if ans in plural_map:
        return plural_map[ans]

    # ── Organ synonyms ──
    organ_map = {
        'liver': 'liver', 'hepatic': 'liver', 'hepatocyte': 'liver',
        'kidney': 'kidney', 'renal': 'kidney',
        'lung': 'lung', 'pulmonary': 'lung',
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'heart': 'heart', 'cardiac': 'heart', 'myocardium': 'heart',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'thyroid': 'thyroid', 'thyroid gland': 'thyroid',
        'stomach': 'stomach', 'gastric': 'stomach',
        'intestine': 'intestine', 'bowel': 'intestine',
        'small intestine': 'small intestine', 'small bowel': 'small intestine',
        'large intestine': 'large intestine', 'large bowel': 'large intestine',
        'colon': 'colon', 'colonic': 'colon',
        'skin': 'skin', 'dermis': 'skin', 'epidermis': 'skin', 'cutaneous': 'skin',
        'bone marrow': 'bone marrow', 'marrow': 'bone marrow',
        'adrenal': 'adrenal gland', 'adrenal gland': 'adrenal gland',
        'pituitary': 'pituitary gland', 'pituitary gland': 'pituitary gland',
        'uterus': 'uterus', 'uterine': 'uterus',
        'ovary': 'ovary', 'ovarian': 'ovary',
        'prostate': 'prostate', 'prostatic': 'prostate',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'esophagus': 'esophagus', 'oesophagus': 'esophagus',
        'trachea': 'trachea', 'tracheal': 'trachea',
        'breast': 'breast', 'mammary': 'breast',
    }
    if ans in organ_map:
        return organ_map[ans]

    # ── Stain normalization ──
    stain_map = {
        'h&e': 'h&e', 'he': 'h&e', 'hematoxylin and eosin': 'h&e',
        'hematoxylin & eosin': 'h&e', 'h and e': 'h&e', 'h & e': 'h&e',
        'pas': 'pas', 'periodic acid-schiff': 'pas',
        'periodic acid schiff': 'pas',
        'masson trichrome': 'trichrome', 'trichrome': 'trichrome',
        'massons trichrome': 'trichrome',
        'giemsa': 'giemsa', 'giemsa stain': 'giemsa',
        'gram stain': 'gram', 'gram': 'gram',
        'silver stain': 'silver stain', 'silver': 'silver stain',
        'immunohistochemistry': 'ihc', 'ihc': 'ihc',
    }
    if ans in stain_map:
        return stain_map[ans]

    # ── Color normalization ──
    color_map = {
        'blue': 'blue', 'bluish': 'blue', 'dark blue': 'blue',
        'red': 'red', 'reddish': 'red', 'dark red': 'red',
        'pink': 'pink', 'pinkish': 'pink',
        'purple': 'purple', 'violet': 'purple', 'purplish': 'purple',
        'brown': 'brown', 'brownish': 'brown', 'dark brown': 'brown',
        'yellow': 'yellow', 'yellowish': 'yellow',
        'white': 'white', 'whitish': 'white',
        'black': 'black', 'blackish': 'black', 'dark': 'black',
        'green': 'green', 'greenish': 'green',
    }
    if ans in color_map:
        return color_map[ans]

    # ── Remove trailing articles ──
    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans



# =============================================================================
# LOAD DATASET INTO RAM
# =============================================================================
from datasets import load_dataset
ds = load_dataset('flaviagiammarino/path-vqa')

def extract(split_data, name):
    samples = []
    for s in tqdm(split_data, desc=name):
        try:
            img = s.get('image'); q = str(s.get('question','')); a = str(s.get('answer','')).strip().lower()
            a = normalize_answer_pathvqa(a)
            if img and q and a:
                img_np = np.array(img.convert('RGB').resize((224,224)), dtype=np.float32) / 255.0
                samples.append({'image': img_np, 'question': q, 'answer': a})
        except: continue
    print(f"  {name}: {len(samples)}"); return samples

train_samples = extract(ds['train'], 'train'); test_samples = extract(ds['test'], 'test'); del ds

all_ans = [s['answer'] for s in train_samples + test_samples]; counts = Counter(all_ans)
answer_vocab = {'<unk>': 0}
for i, a in enumerate(sorted(set(all_ans))): answer_vocab[a] = i + 1
num_classes = len(answer_vocab)
print(f"  Vocab: {num_classes} classes")

indices = list(range(len(train_samples))); random.shuffle(indices)
n_val = int(len(indices) * VAL_SPLIT)
trn = [train_samples[i] for i in indices[n_val:]]
val = [train_samples[i] for i in indices[:n_val]]
print(f"  Train: {len(trn)}, Val: {len(val)}, Test: {len(test_samples)}")

# =============================================================================
# TOKENIZER (hash-based, matching TF version)
# =============================================================================
def tokenize(questions):
    ids_list, mask_list = [], []
    for q in questions:
        words = q.lower().split()[:MAX_SEQ-2]
        ids = [1] + [hash(w) % (VOCAB_SIZE-2) + 2 for w in words] + [2]
        mask = [1.0] * len(ids)
        while len(ids) < MAX_SEQ: ids.append(0); mask.append(0.0)
        ids_list.append(ids[:MAX_SEQ]); mask_list.append(mask[:MAX_SEQ])
    return ids_list, mask_list

# =============================================================================
# DATASET
# =============================================================================
class VQADataset(Dataset):
    def __init__(self, samples, vocab, augment=False):
        self.samples = samples; self.vocab = vocab; self.augment = augment
        all_qs = [s['question'] for s in samples]
        self.ids, self.masks = tokenize(all_qs)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = torch.tensor(s['image']).permute(2,0,1)  # HWC → CHW
        if self.augment:
            if random.random() > 0.5: img = img.flip(-1)
            img = img + torch.randn_like(img) * 0.02
            img = img.clamp(0,1)
        ids = torch.tensor(self.ids[idx], dtype=torch.long)
        mask = torch.tensor(self.masks[idx], dtype=torch.float32)
        lbl = self.vocab.get(s['answer'], 0)
        return img, ids, mask, lbl

def collate_fn(batch):
    imgs, ids, masks, lbls = zip(*batch)
    return torch.stack(imgs), torch.stack(ids), torch.stack(masks), torch.tensor(lbls, dtype=torch.long)

train_loader = DataLoader(VQADataset(trn, answer_vocab, augment=True), batch_size=BS, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(VQADataset(val, answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader = DataLoader(VQADataset(test_samples, answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

# =============================================================================
# MODEL BLOCKS (PyTorch port of the TF/Keras architecture)
# =============================================================================

class TransformerBlock(nn.Module):
    """Pre-norm transformer block."""
    def __init__(self, dim, n_heads=4, ffn_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim); self.norm2 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*ffn_ratio), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(dim*ffn_ratio, dim), nn.Dropout(dropout))
    def forward(self, x, mask=None):
        h = self.norm1(x)
        kpm = (mask == 0) if mask is not None else None
        h, _ = self.attn(h, h, h, key_padding_mask=kpm)
        x = x + h
        return x + self.ffn(self.norm2(x))


class VisionEncoder(nn.Module):
    """Lightweight CNN: 224×224×3 → 49 tokens × D. ~0.8M params."""
    def __init__(self, dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, dim, 3, stride=2, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        h = self.pool1(F.silu(self.bn1(self.conv1(x))))
        h = F.silu(self.bn2(self.conv2(h)))
        h = F.silu(self.bn3(self.conv3(h)))
        h = F.silu(self.bn4(self.conv4(h)))
        B, C, H, W = h.shape
        return self.norm(h.permute(0,2,3,1).reshape(B, H*W, C))  # (B,49,D)


class TextEncoder(nn.Module):
    """Embedding + Transformer layers. ~0.7M params."""
    def __init__(self, vocab_size=30522, dim=256, n_layers=2, n_heads=4, max_len=64, dropout=0.1):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, dim)
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, dim) * 0.02)
        self.embed_norm = nn.LayerNorm(dim)
        self.embed_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads, dropout=dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(dim)
    def forward(self, input_ids, mask=None):
        L = input_ids.shape[1]
        x = self.tok_embed(input_ids) + self.pos_embed[:, :L, :]
        x = self.embed_drop(self.embed_norm(x))
        for blk in self.blocks: x = blk(x, mask=mask)
        return self.final_norm(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, ratio=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels//ratio, bias=False)
        self.fc2 = nn.Linear(channels//ratio, channels, bias=False)
    def forward(self, x):
        # x: (B,H,W,C)
        avg = x.mean(dim=[1,2], keepdim=True); mx = x.amax(dim=[1,2], keepdim=True)
        return x * torch.sigmoid(self.fc2(F.silu(self.fc1(avg))) + self.fc2(F.silu(self.fc1(mx))))


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 8, 3, padding=1, bias=False)
        self.conv2 = nn.Conv2d(2, 8, 3, padding=2, dilation=2, bias=False)
        self.fuse = nn.Conv2d(16, 1, 1, bias=False)
    def forward(self, x):
        # x: (B,H,W,C) → need (B,C,H,W) for conv
        xp = x.permute(0,3,1,2)
        avg = xp.mean(dim=1, keepdim=True); mx = xp.amax(dim=1, keepdim=True)
        cat = torch.cat([avg, mx], dim=1)
        ms = torch.cat([self.conv1(cat), self.conv2(cat)], dim=1)
        attn = torch.sigmoid(self.fuse(ms)).permute(0,2,3,1)
        return x * attn


class CBAMBlock(nn.Module):
    """CBAM on tokens: reshape (B,49,D) → (B,7,7,D), apply CBAM, back."""
    def __init__(self, channels):
        super().__init__()
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()
        self.ffn = nn.Sequential(nn.Linear(channels, channels*2), nn.GELU(), nn.Linear(channels*2, channels))
        self.norm1 = nn.LayerNorm(channels); self.norm2 = nn.LayerNorm(channels)
    def forward(self, tokens):
        B, N, C = tokens.shape
        spatial = tokens.reshape(B, 7, 7, C)
        spatial = self.sa(self.ca(spatial))
        refined = spatial.reshape(B, N, C)
        tokens = self.norm1(tokens + refined)
        return self.norm2(tokens + self.ffn(tokens))


class FusionLayer(nn.Module):
    """Bidirectional cross-attention fusion."""
    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        self.v2t = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.v2t_norm = nn.LayerNorm(dim)
        self.v2t_ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*4, dim))
        self.v2t_ffn_norm = nn.LayerNorm(dim)
        self.t2v = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.t2v_norm = nn.LayerNorm(dim)
        self.t2v_ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*4, dim))
        self.t2v_ffn_norm = nn.LayerNorm(dim)
    def forward(self, v, t, text_kpm=None):
        o, _ = self.v2t(v, t, t, key_padding_mask=text_kpm)
        v = self.v2t_norm(v + o); v = self.v2t_ffn_norm(v + self.v2t_ffn(v))
        o, _ = self.t2v(t, v, v)
        t = self.t2v_norm(t + o); t = self.t2v_ffn_norm(t + self.t2v_ffn(t))
        return v, t


# =============================================================================
# FULL MODEL
# =============================================================================
class MedicalVQAModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Encoders
        self.vision_enc = VisionEncoder(D)
        self.text_enc = TextEncoder(VOCAB_SIZE, D, TEXT_ENC_LAYERS, HEADS, MAX_SEQ, DROP)

        # CBAM visual refinement
        self.vis_refine = nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)])

        # Text refinement
        self.text_refine = nn.ModuleList([TransformerBlock(D, HEADS, dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)])
        self.text_refine_norm = nn.LayerNorm(D)

        # Question-aware visual attention
        self.q_attn = nn.MultiheadAttention(D, HEADS, dropout=DROP, batch_first=True)
        self.q_gate = nn.Linear(D, D)
        self.q_norm = nn.LayerNorm(D)

        # Cross-attention fusion
        self.fusion_layers = nn.ModuleList([FusionLayer(D, HEADS, DROP) for _ in range(FUSE_LAYERS)])

        # Learnable pooling
        self.pool_query = nn.Parameter(torch.randn(1, 1, D) * 0.02)
        self.pool_attn = nn.MultiheadAttention(D, HEADS, dropout=DROP, batch_first=True)
        self.pool_norm = nn.LayerNorm(D)

        # Classification head with residual
        self.head_fc1 = nn.Linear(D, D)
        self.head_drop1 = nn.Dropout(DROP)
        self.head_fc2 = nn.Linear(D, D//2)
        self.head_drop2 = nn.Dropout(DROP)
        self.head_out = nn.Linear(D//2, num_classes)
        self.head_res = nn.Linear(D, D//2)
        self.head_norm = nn.LayerNorm(D//2)

    def forward(self, images, input_ids, text_mask):
        # 1. Encode
        v = self.vision_enc(images)                                   # (B,49,D)
        t = self.text_enc(input_ids, mask=text_mask)                  # (B,L,D)

        # 2. CBAM visual refinement
        for blk in self.vis_refine: v = blk(v)

        # 3. Text refinement
        for blk in self.text_refine: t = blk(t, mask=text_mask)
        t = self.text_refine_norm(t)

        # 4. Question-aware visual attention
        q_cls = t[:, 0:1, :].expand(-1, v.shape[1], -1)
        attn_out, _ = self.q_attn(q_cls, v, v)
        gate = torch.sigmoid(self.q_gate(attn_out))
        v = self.q_norm(v + v * gate + attn_out * (1.0 - gate))

        # 5. Cross-attention fusion
        text_kpm = (text_mask == 0)
        for fl in self.fusion_layers: v, t = fl(v, t, text_kpm=text_kpm)

        # 6. Pool
        combined = torch.cat([v, t], dim=1)
        B = combined.shape[0]
        pq = self.pool_query.expand(B, -1, -1)
        pooled, _ = self.pool_attn(pq, combined, combined)
        fused = self.pool_norm(pq + pooled).squeeze(1)              # (B,D)

        # 7. Head with residual
        h = self.head_drop1(F.gelu(self.head_fc1(fused)))
        h = self.head_drop2(F.gelu(self.head_fc2(h)))
        h = self.head_norm(h + self.head_res(fused))
        return self.head_out(h)

model = MedicalVQAModel(num_classes).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"\n  Model: {n_params:,} params ({n_params/1e6:.1f}M)")

# =============================================================================
# TRAINING
# =============================================================================
print("\n" + "="*60 + "\nTRAINING (Centralized)\n" + "="*60)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

history = {'epoch':[],'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],
           'test_loss':[],'test_acc':[],'lr':[],'epoch_time':[]}
best_val, best_state, pat, lr_pat = 0.0, None, 0, 0

@torch.no_grad()
def eval_loader(loader):
    model.eval(); ls, c, t = 0.0, 0, 0
    for imgs, ids, masks, lbls in loader:
        imgs, ids, masks, lbls = imgs.to(device), ids.to(device), masks.to(device), lbls.to(device)
        logits = model(imgs, ids, masks); loss = criterion(logits, lbls)
        ls += loss.item()*lbls.size(0); c += (logits.argmax(-1)==lbls).sum().item(); t += lbls.size(0)
    return ls/t, 100*c/t

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    model.train(); tr_l, tr_c, tr_t = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"E{epoch:02d}/{EPOCHS}", leave=False)
    for imgs, ids, masks, lbls in pbar:
        imgs, ids, masks, lbls = imgs.to(device), ids.to(device), masks.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = model(imgs, ids, masks); loss = criterion(logits, lbls)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        tr_l += loss.item()*lbls.size(0); tr_c += (logits.argmax(-1)==lbls).sum().item(); tr_t += lbls.size(0)
        pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{100*tr_c/tr_t:.1f}%")
    tr_l /= tr_t; tr_a = 100*tr_c/tr_t

    va_l, va_a = eval_loader(val_loader); te_l, te_a = eval_loader(test_loader)
    lr = optimizer.param_groups[0]['lr']; et = time.time()-t0

    history['epoch'].append(epoch); history['train_loss'].append(tr_l); history['train_acc'].append(tr_a)
    history['val_loss'].append(va_l); history['val_acc'].append(va_a)
    history['test_loss'].append(te_l); history['test_acc'].append(te_a)
    history['lr'].append(lr); history['epoch_time'].append(et)

    mk = ""
    if va_a > best_val:
        best_val = va_a; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        pat = lr_pat = 0; mk = " ★"
    else: pat += 1; lr_pat += 1
    print(f"E{epoch:02d} [{et:.1f}s]  Train: {tr_l:.4f}/{tr_a:.1f}%  Val: {va_l:.4f}/{va_a:.1f}%  Test: {te_a:.1f}%  LR={lr:.1e}{mk}")

    if lr_pat >= LR_PAT:
        for pg in optimizer.param_groups: pg['lr'] = max(pg['lr']*LR_FAC, MIN_LR)
        lr_pat = 0
    if pat >= ES_PAT: print(f"  Early stop at epoch {epoch}"); break

if best_state: model.load_state_dict(best_state)
te_l, te_a = eval_loader(test_loader)
print(f"\n{'='*60}\nFINAL TEST: {te_a:.2f}%\n{'='*60}")

# =============================================================================
# SAVE EXCEL
# =============================================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

wb = openpyxl.Workbook(); ws = wb.active; ws.title = "Training"
hf = Font(name='Arial', bold=True, size=11, color='FFFFFF')
hfi = PatternFill(start_color='1A5276', end_color='1A5276', fill_type='solid')
headers = ['Epoch','Train Loss','Train Acc (%)','Val Loss','Val Acc (%)','Test Loss','Test Acc (%)','LR','Time (s)']
for c, h in enumerate(headers, 1):
    cl = ws.cell(row=1, column=c, value=h); cl.font = hf; cl.fill = hfi; cl.alignment = Alignment(horizontal='center')
for i, ep in enumerate(history['epoch']):
    r = i+2
    for c, k in enumerate(['epoch','train_loss','train_acc','val_loss','val_acc','test_loss','test_acc','lr','epoch_time'], 1):
        v = history[k][i]; ws.cell(row=r, column=c, value=round(v,4) if isinstance(v,float) else v)

ws2 = wb.create_sheet("Summary")
for i, (k,v) in enumerate([("Method","Centralized"),("Dataset","VQA-RAD"),
    ("Architecture","CNN+CBAM+CrossAttn (custom)"),("Params",f"{n_params:,}"),
    ("Classes",num_classes),("Best Val",round(best_val,2)),("Final Test",round(te_a,2)),
    ("Epochs",len(history['epoch'])),("Batch Size",BS),("LR",LR)], 1):
    ws2.cell(row=i, column=1, value=k).font = Font(bold=True, name='Arial'); ws2.cell(row=i, column=2, value=v)
for s in [ws, ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width = max(len(str(c.value or '')) for c in col)+2
p = f"{OUTPUT_DIR}/pathvqa_centralized_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")